In [ ]:
"""
"Red Flags" Hyperparameter Analysis
====================================
Goal: Identify hyperparameter settings that *underperform* across
multiple tasks.

Data format: WIDE — one row per model, one column per task metric.

The analysis is run TWICE:
  • GROUPED   — metrics are averaged within each HP group before ranking,
                so every unique HP configuration counts once per task.
  • UNGROUPED — every individual model row is ranked independently,
                reflecting within-group variance as well.

Hyperparameters : lower, mincount, windows
Intrinsic tasks : bestmatch, opposite, wordintrusion, mostsimilar
Extrinsic tasks : ffp, topics, autnes_sentiment
"""

import pandas as pd
import numpy as np
from scipy import stats
import itertools
import warnings
warnings.filterwarnings("ignore")

# ── 0. CONFIG ──────────────────────────────────────────────────────────────────
DATA_PATH = "dataset_regression.csv"  # ← path to the regression dataset (one row per model, one column per task)
HP_COLS   = ["lower", "mincount", "windows"]     # hyperparameter columns

TASK_LABELS = {
    "bestmatch":        "Intrinsic Task 1: Best Match",
    "opposite":         "Intrinsic Task 2: Opposite",
    "wordintrusion":    "Intrinsic Task 3: Word Intrusion",
    "mostsimilar":      "Intrinsic Task 4: Grammar",
    "ffp":              "Extrinsic Task 1: Author Prediction",
    "topics":           "Extrinsic Task 2: Topic Prediction",
    "autnes_sentiment": "Extrinsic Task 3: Sentiment Prediction",
}

INTRINSIC_TASKS = ["bestmatch", "opposite", "wordintrusion", "mostsimilar"]
EXTRINSIC_TASKS = ["ffp", "topics", "autnes_sentiment"]
ALL_TASKS       = INTRINSIC_TASKS + EXTRINSIC_TASKS

LOWER_IS_BETTER       = False   # set True if lower metric = better
ALPHA                 = 0.05    # significance level
RED_FLAG_TASK_FRACTION = 0.6   # flag if bottom-25 % in >= 60 % of tasks

TASK_COL   = "task"
METRIC_COL = "metric"

# ── 1. LOAD & PREPARE ──────────────────────────────────────────────────────────
df_wide = pd.read_csv(DATA_PATH)
df_wide = df_wide.iloc[:-4].copy()   # exclude last 4 baseline/reference models

# Exclude specific models by name
MODEL_COL     = "name"             # column holding the model identifier
EXCLUDE_EXACT = [                  # exact model names to drop
    "wiki_de_300_wikipedia_lr0.1_epochs5_mincount1_ws5_dims300",
]
EXCLUDE_CONTAINS = ["cc_de", "wiki_de"]  # drop any model whose name contains these

before = len(df_wide)
exact_mask    = df_wide[MODEL_COL].isin(EXCLUDE_EXACT)
contains_mask = df_wide[MODEL_COL].str.contains("|".join(EXCLUDE_CONTAINS), na=False)
df_wide = df_wide[~(exact_mask | contains_mask)].copy()
print(f"Rows after all model exclusions: {len(df_wide)}  (dropped {before - len(df_wide)})")

missing_hp   = [c for c in HP_COLS   if c not in df_wide.columns]
missing_task = [c for c in ALL_TASKS if c not in df_wide.columns]
if missing_hp:
    raise ValueError(f"HP columns not found in CSV: {missing_hp}")
if missing_task:
    print(f"WARNING — task columns not found in CSV (skipped): {missing_task}")

available_tasks = [t for t in ALL_TASKS if t in df_wide.columns]
ID_VARS         = ["group_number"] + HP_COLS

# Melt to long format (one row per model × task)
df_long = df_wide.melt(
    id_vars    = ID_VARS,
    value_vars = available_tasks,
    var_name   = TASK_COL,
    value_name = METRIC_COL,
).dropna(subset=[METRIC_COL])

if LOWER_IS_BETTER:
    df_long[METRIC_COL] = -df_long[METRIC_COL]

# ── Two analysis dataframes ────────────────────────────────────────────────────
# GROUPED: average metrics within each (group, task) → one row per HP config × task
df_grouped = (
    df_long
    .groupby(ID_VARS + [TASK_COL], as_index=False)[METRIC_COL]
    .mean()
)

# UNGROUPED: keep every individual model row
df_ungrouped = df_long.copy()

found_tasks = sorted(df_long[TASK_COL].unique())
print(f"HP configurations (groups) : {df_grouped['group_number'].nunique()}")
print(f"Individual model rows       : {len(df_ungrouped)}")
print(f"Tasks                       : {len(found_tasks)}\n")


# ══════════════════════════════════════════════════════════════════════════════
#  CORE ANALYSIS FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def run_analysis(df: pd.DataFrame, label: str, output_prefix: str) -> None:
    """
    Full red-flag analysis on a (long-format) dataframe.

    Parameters
    ----------
    df            : long-format dataframe with HP_COLS, TASK_COL, METRIC_COL
    label         : human-readable name shown in section headers
    output_prefix : prefix for saved CSV files (e.g. 'grouped' or 'ungrouped')
    """

    sep = "═" * 70
    print(f"\n{sep}")
    print(f"  ANALYSIS: {label.upper()}")
    print(f"{sep}\n")

    df = df.copy()
    ft = sorted(df[TASK_COL].unique())   # found tasks for this run

    # ── Step 2: Per-task percentile rank ──────────────────────────────────────
    df["rank_within_task"] = (
        df.groupby(TASK_COL)[METRIC_COL]
          .rank(pct=True)
    )

    # ── Step 3: Aggregate rank per HP setting ─────────────────────────────────
    results = {}
    for hp in HP_COLS:
        grouped_hp = (
            df.groupby([hp, TASK_COL])["rank_within_task"]
              .mean()
              .reset_index(name="mean_rank")
        )
        grouped_hp["task_type"] = grouped_hp[TASK_COL].apply(
            lambda t: "intrinsic" if t in INTRINSIC_TASKS else "extrinsic"
        )

        def agg_summary(sub):
            intr = sub[sub["task_type"] == "intrinsic"]["mean_rank"]
            extr = sub[sub["task_type"] == "extrinsic"]["mean_rank"]
            return pd.Series({
                "mean_rank":            sub["mean_rank"].mean(),
                "n_tasks":              sub[TASK_COL].nunique(),
                "n_tasks_bottom25":     (sub["mean_rank"] < 0.25).sum(),
                "mean_rank_intrinsic":  intr.mean() if len(intr) else np.nan,
                "n_intrinsic_bottom25": (intr < 0.25).sum(),
                "mean_rank_extrinsic":  extr.mean() if len(extr) else np.nan,
                "n_extrinsic_bottom25": (extr < 0.25).sum(),
            })

        summary = grouped_hp.groupby(hp).apply(agg_summary).reset_index()
        summary["frac_tasks_bottom25"] = (
            summary["n_tasks_bottom25"] / summary["n_tasks"]
        )
        results[hp] = summary.sort_values("mean_rank")

    # ── Step 4: Statistical significance ─────────────────────────────────────
    def run_kw_posthoc(df_sub, hp, sublabel):
        groups = {
            val: df_sub.loc[df_sub[hp] == val, "rank_within_task"].values
            for val in df_sub[hp].unique()
        }
        if len(groups) < 2:
            return pd.DataFrame()
        stat, p_kw = stats.kruskal(*groups.values())
        if p_kw >= ALPHA:
            print(f"    [{hp} | {sublabel}] KW p={p_kw:.3f} → no significant difference.")
            return pd.DataFrame()
        print(f"    [{hp} | {sublabel}] KW p={p_kw:.4f} ✓ → pairwise tests …")
        best_val  = max(groups, key=lambda v: groups[v].mean())
        best_data = groups[best_val]
        rows = []
        for val, other in groups.items():
            if val == best_val:
                continue
            _, p_mw = stats.mannwhitneyu(best_data, other, alternative="greater")
            rows.append({
                hp: val, "subset": sublabel,
                "p_vs_best": round(p_mw, 4),
                "significantly_worse": p_mw < ALPHA,
            })
        return pd.DataFrame(rows)

    pairwise_flags = {}
    print("── Statistical tests ──")
    for hp in HP_COLS:
        print(f"  {hp}:")
        frames = [
            run_kw_posthoc(df,                                      hp, "all tasks"),
            run_kw_posthoc(df[df[TASK_COL].isin(INTRINSIC_TASKS)], hp, "intrinsic"),
            run_kw_posthoc(df[df[TASK_COL].isin(EXTRINSIC_TASKS)], hp, "extrinsic"),
        ]
        non_empty = [f for f in frames if not f.empty]
        pairwise_flags[hp] = (
            pd.concat(non_empty, ignore_index=True)
            if non_empty else pd.DataFrame()
        )

    # ── Step 5: Per-task heatmap ───────────────────────────────────────────────
    print(f"\n── Per-task rank heatmap ──")
    for hp in HP_COLS:
        pivot = (
            df.groupby([hp, TASK_COL])["rank_within_task"]
              .mean()
              .reset_index(name="mean_rank")
              .pivot(index=hp, columns=TASK_COL, values="mean_rank")
              .reindex(columns=[t for t in ALL_TASKS if t in ft])
              .rename(columns=TASK_LABELS)
              .round(3)
        )
        fname = f"heatmap_{output_prefix}_{hp}.csv"
        pivot.to_csv(fname)
        print(f"  {hp}:\n{pivot.to_string()}\n  → Saved to {fname}\n")

    # ── Step 6: Red flag summary ───────────────────────────────────────────────
    print(f"── Red flag summary ──")
    red_flag_records = []

    for hp in HP_COLS:
        summ = results[hp]
        pw   = pairwise_flags.get(hp, pd.DataFrame())

        sig_worse_vals = set()
        if not pw.empty:
            sig_worse_vals = set(pw.loc[pw["significantly_worse"], hp].tolist())

        for _, row in summ.iterrows():
            val         = row[hp]
            frac_bad    = row["frac_tasks_bottom25"]
            sig_worse   = val in sig_worse_vals
            is_red_flag = (frac_bad >= RED_FLAG_TASK_FRACTION) or sig_worse

            if not is_red_flag:
                continue

            bad_task_names = [
                TASK_LABELS.get(t, t) for t in ft
                if df.loc[(df[TASK_COL] == t) & (df[hp] == val),
                           "rank_within_task"].mean() < 0.25
            ]
            red_flag_records.append({
                "hyperparameter":      hp,
                "value":               val,
                "mean_rank_all":       round(row["mean_rank"], 3),
                "mean_rank_intrinsic": (round(row["mean_rank_intrinsic"], 3)
                                        if not np.isnan(row["mean_rank_intrinsic"]) else "–"),
                "mean_rank_extrinsic": (round(row["mean_rank_extrinsic"], 3)
                                        if not np.isnan(row["mean_rank_extrinsic"]) else "–"),
                "frac_tasks_bottom25": round(frac_bad, 2),
                "n_tasks_bottom25":    int(row["n_tasks_bottom25"]),
                "stat_sig_worse":      sig_worse,
                "bottom25_in_tasks":   "; ".join(bad_task_names) if bad_task_names else "–",
            })

    if red_flag_records:
        rf_df = (
            pd.DataFrame(red_flag_records)
              .sort_values(["frac_tasks_bottom25", "mean_rank_all"],
                           ascending=[False, True])
        )
        print(rf_df.to_string(index=False))
        fname = f"red_flags_{output_prefix}.csv"
        rf_df.to_csv(fname, index=False)
        print(f"\n→ Saved to {fname}")
    else:
        print("  No red-flag settings identified at the current thresholds.")

    # ── Step 7: Consistently bad in BOTH task types ───────────────────────────
    print(f"\n── Consistently bad in BOTH intrinsic AND extrinsic ──")
    for hp in HP_COLS:
        summ     = results[hp]
        both_bad = summ[
            (summ["n_intrinsic_bottom25"] >=
             len(INTRINSIC_TASKS) * RED_FLAG_TASK_FRACTION) &
            (summ["n_extrinsic_bottom25"] >=
             len(EXTRINSIC_TASKS) * RED_FLAG_TASK_FRACTION)
        ]
        if not both_bad.empty:
            print(f"  {hp}:")
            print(both_bad[[hp, "mean_rank", "mean_rank_intrinsic",
                             "mean_rank_extrinsic",
                             "frac_tasks_bottom25"]].to_string(index=False))
        else:
            print(f"  {hp}: no value consistently bottom-25 % in both task types.")

    # ── Step 8: Pairwise HP interaction red flags ──────────────────────────────
    print(f"\n── Interaction red flags (HP pairs) ──")
    for hp_a, hp_b in itertools.combinations(HP_COLS, 2):
        combo_col = f"{hp_a}__{hp_b}"
        df_tmp    = df.copy()
        df_tmp[combo_col] = (df_tmp[hp_a].astype(str)
                             + " | " + df_tmp[hp_b].astype(str))

        combo_summary = (
            df_tmp.groupby([combo_col, TASK_COL])["rank_within_task"]
              .mean().reset_index()
              .groupby(combo_col)
              .agg(
                  mean_rank        = ("rank_within_task", "mean"),
                  n_tasks          = (TASK_COL,           "nunique"),
                  n_tasks_bottom25 = ("rank_within_task",
                                      lambda x: (x < 0.25).sum()),
              )
              .assign(frac_tasks_bottom25=lambda d: d["n_tasks_bottom25"] / d["n_tasks"])
              .reset_index()
        )
        bad = combo_summary[
            combo_summary["frac_tasks_bottom25"] >= RED_FLAG_TASK_FRACTION
        ].sort_values("frac_tasks_bottom25", ascending=False)

        if not bad.empty:
            print(f"  Pair ({hp_a}, {hp_b}):")
            print(bad[[combo_col, "mean_rank",
                        "frac_tasks_bottom25", "n_tasks"]].to_string(index=False))
        else:
            print(f"  Pair ({hp_a}, {hp_b}): no red-flag combinations found.")


# ══════════════════════════════════════════════════════════════════════════════
#  RUN BOTH ANALYSES
# ══════════════════════════════════════════════════════════════════════════════

run_analysis(
    df            = df_grouped,
    label         = "Grouped (one data point per HP configuration per task)",
    output_prefix = "grouped",
)

run_analysis(
    df            = df_ungrouped,
    label         = "Ungrouped (individual model rows, within-group variance retained)",
    output_prefix = "ungrouped",
)

print("\n✓ Both analyses complete.")


Rows after all model exclusions: 320  (dropped 2)
HP configurations (groups) : 32
Individual model rows       : 2240
Tasks                       : 7


══════════════════════════════════════════════════════════════════════
  ANALYSIS: GROUPED (ONE DATA POINT PER HP CONFIGURATION PER TASK)
══════════════════════════════════════════════════════════════════════

── Statistical tests ──
  lower:
    [lower | all tasks] KW p=0.860 → no significant difference.
    [lower | intrinsic] KW p=0.110 → no significant difference.
    [lower | extrinsic] KW p=0.118 → no significant difference.
  mincount:
    [mincount | all tasks] KW p=0.0001 ✓ → pairwise tests …
    [mincount | intrinsic] KW p=0.0000 ✓ → pairwise tests …
    [mincount | extrinsic] KW p=0.0035 ✓ → pairwise tests …
  windows:
    [windows | all tasks] KW p=0.0089 ✓ → pairwise tests …
    [windows | intrinsic] KW p=0.153 → no significant difference.
    [windows | extrinsic] KW p=0.076 → no significant difference.

── Per-task rank he

In [ ]:
"""
For ROBUSTNESS: WITHOUT GRAMMAR 
"Red Flags" Hyperparameter Analysis
====================================
Goal: Identify hyperparameter settings that  underperform across
multiple tasks

Data format: WIDE — one row per model, one column per task metric.

The analysis is run TWICE:
  • GROUPED   — metrics are averaged within each HP group before ranking,
                so every unique HP configuration counts once per task.
  • UNGROUPED — every individual model row is ranked independently,
                reflecting within-group variance as well.

Hyperparameters : lower, mincount, windows
Intrinsic tasks : bestmatch, opposite, wordintrusion, mostsimilar
Extrinsic tasks : ffp, topics, autnes_sentiment
"""

import pandas as pd
import numpy as np
from scipy import stats
import itertools
import warnings
warnings.filterwarnings("ignore")

# ── 0. CONFIG ──────────────────────────────────────────────────────────────────
DATA_PATH = "dataset_regression.csv"  # ← path to the regression dataset (one row per model, one column per task)
HP_COLS   = ["lower", "mincount", "windows"]     # hyperparameter columns

TASK_LABELS = {
    "bestmatch":        "Intrinsic Task 1: Best Match",
    "opposite":         "Intrinsic Task 2: Opposite",
    "wordintrusion":    "Intrinsic Task 3: Word Intrusion",
    "ffp":              "Extrinsic Task 1: Author Prediction",
    "topics":           "Extrinsic Task 2: Topic Prediction",
    "autnes_sentiment": "Extrinsic Task 3: Sentiment Prediction",
}

INTRINSIC_TASKS = ["bestmatch", "opposite", "wordintrusion"]
EXTRINSIC_TASKS = ["ffp", "topics", "autnes_sentiment"]
ALL_TASKS       = INTRINSIC_TASKS + EXTRINSIC_TASKS

LOWER_IS_BETTER       = False   # set True if lower metric = better
ALPHA                 = 0.05    # significance level
RED_FLAG_TASK_FRACTION = 0.6   # flag if bottom-25 % in >= 60 % of tasks

TASK_COL   = "task"
METRIC_COL = "metric"

# ── 1. LOAD & PREPARE ──────────────────────────────────────────────────────────
df_wide = pd.read_csv(DATA_PATH)
df_wide = df_wide.iloc[:-4].copy()   # exclude last 4 baseline/reference models

# Exclude specific models by name
MODEL_COL     = "name"             # column holding the model identifier
EXCLUDE_EXACT = [                  # exact model names to drop
    "wiki_de_300_wikipedia_lr0.1_epochs5_mincount1_ws5_dims300",
]
EXCLUDE_CONTAINS = ["cc_de", "wiki_de"]  # drop any model whose name contains these

before = len(df_wide)
exact_mask    = df_wide[MODEL_COL].isin(EXCLUDE_EXACT)
contains_mask = df_wide[MODEL_COL].str.contains("|".join(EXCLUDE_CONTAINS), na=False)
df_wide = df_wide[~(exact_mask | contains_mask)].copy()
print(f"Rows after all model exclusions: {len(df_wide)}  (dropped {before - len(df_wide)})")

missing_hp   = [c for c in HP_COLS   if c not in df_wide.columns]
missing_task = [c for c in ALL_TASKS if c not in df_wide.columns]
if missing_hp:
    raise ValueError(f"HP columns not found in CSV: {missing_hp}")
if missing_task:
    print(f"WARNING — task columns not found in CSV (skipped): {missing_task}")

available_tasks = [t for t in ALL_TASKS if t in df_wide.columns]
ID_VARS         = ["group_number"] + HP_COLS

# Melt to long format (one row per model × task)
df_long = df_wide.melt(
    id_vars    = ID_VARS,
    value_vars = available_tasks,
    var_name   = TASK_COL,
    value_name = METRIC_COL,
).dropna(subset=[METRIC_COL])

if LOWER_IS_BETTER:
    df_long[METRIC_COL] = -df_long[METRIC_COL]

# ── Two analysis dataframes ────────────────────────────────────────────────────
# GROUPED: average metrics within each (group, task) → one row per HP config × task
df_grouped = (
    df_long
    .groupby(ID_VARS + [TASK_COL], as_index=False)[METRIC_COL]
    .mean()
)

# UNGROUPED: keep every individual model row
df_ungrouped = df_long.copy()

found_tasks = sorted(df_long[TASK_COL].unique())
print(f"HP configurations (groups) : {df_grouped['group_number'].nunique()}")
print(f"Individual model rows       : {len(df_ungrouped)}")
print(f"Tasks                       : {len(found_tasks)}\n")


# ══════════════════════════════════════════════════════════════════════════════
#  CORE ANALYSIS FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def run_analysis(df: pd.DataFrame, label: str, output_prefix: str) -> None:
    """
    Full red-flag analysis on a (long-format) dataframe.

    Parameters
    ----------
    df            : long-format dataframe with HP_COLS, TASK_COL, METRIC_COL
    label         : human-readable name shown in section headers
    output_prefix : prefix for saved CSV files (e.g. 'grouped' or 'ungrouped')
    """

    sep = "═" * 70
    print(f"\n{sep}")
    print(f"  ANALYSIS: {label.upper()}")
    print(f"{sep}\n")

    df = df.copy()
    ft = sorted(df[TASK_COL].unique())   # found tasks for this run

    # ── Step 2: Per-task percentile rank ──────────────────────────────────────
    df["rank_within_task"] = (
        df.groupby(TASK_COL)[METRIC_COL]
          .rank(pct=True)
    )

    # ── Step 3: Aggregate rank per HP setting ─────────────────────────────────
    results = {}
    for hp in HP_COLS:
        grouped_hp = (
            df.groupby([hp, TASK_COL])["rank_within_task"]
              .mean()
              .reset_index(name="mean_rank")
        )
        grouped_hp["task_type"] = grouped_hp[TASK_COL].apply(
            lambda t: "intrinsic" if t in INTRINSIC_TASKS else "extrinsic"
        )

        def agg_summary(sub):
            intr = sub[sub["task_type"] == "intrinsic"]["mean_rank"]
            extr = sub[sub["task_type"] == "extrinsic"]["mean_rank"]
            return pd.Series({
                "mean_rank":            sub["mean_rank"].mean(),
                "n_tasks":              sub[TASK_COL].nunique(),
                "n_tasks_bottom25":     (sub["mean_rank"] < 0.25).sum(),
                "mean_rank_intrinsic":  intr.mean() if len(intr) else np.nan,
                "n_intrinsic_bottom25": (intr < 0.25).sum(),
                "mean_rank_extrinsic":  extr.mean() if len(extr) else np.nan,
                "n_extrinsic_bottom25": (extr < 0.25).sum(),
            })

        summary = grouped_hp.groupby(hp).apply(agg_summary).reset_index()
        summary["frac_tasks_bottom25"] = (
            summary["n_tasks_bottom25"] / summary["n_tasks"]
        )
        results[hp] = summary.sort_values("mean_rank")

    # ── Step 4: Statistical significance ─────────────────────────────────────
    def run_kw_posthoc(df_sub, hp, sublabel):
        groups = {
            val: df_sub.loc[df_sub[hp] == val, "rank_within_task"].values
            for val in df_sub[hp].unique()
        }
        if len(groups) < 2:
            return pd.DataFrame()
        stat, p_kw = stats.kruskal(*groups.values())
        if p_kw >= ALPHA:
            print(f"    [{hp} | {sublabel}] KW p={p_kw:.3f} → no significant difference.")
            return pd.DataFrame()
        print(f"    [{hp} | {sublabel}] KW p={p_kw:.4f} ✓ → pairwise tests …")
        best_val  = max(groups, key=lambda v: groups[v].mean())
        best_data = groups[best_val]
        rows = []
        for val, other in groups.items():
            if val == best_val:
                continue
            _, p_mw = stats.mannwhitneyu(best_data, other, alternative="greater")
            rows.append({
                hp: val, "subset": sublabel,
                "p_vs_best": round(p_mw, 4),
                "significantly_worse": p_mw < ALPHA,
            })
        return pd.DataFrame(rows)

    pairwise_flags = {}
    print("── Statistical tests ──")
    for hp in HP_COLS:
        print(f"  {hp}:")
        frames = [
            run_kw_posthoc(df,                                      hp, "all tasks"),
            run_kw_posthoc(df[df[TASK_COL].isin(INTRINSIC_TASKS)], hp, "intrinsic"),
            run_kw_posthoc(df[df[TASK_COL].isin(EXTRINSIC_TASKS)], hp, "extrinsic"),
        ]
        non_empty = [f for f in frames if not f.empty]
        pairwise_flags[hp] = (
            pd.concat(non_empty, ignore_index=True)
            if non_empty else pd.DataFrame()
        )

    # ── Step 5: Per-task heatmap ───────────────────────────────────────────────
    print(f"\n── Per-task rank heatmap ──")
    for hp in HP_COLS:
        pivot = (
            df.groupby([hp, TASK_COL])["rank_within_task"]
              .mean()
              .reset_index(name="mean_rank")
              .pivot(index=hp, columns=TASK_COL, values="mean_rank")
              .reindex(columns=[t for t in ALL_TASKS if t in ft])
              .rename(columns=TASK_LABELS)
              .round(3)
        )
        fname = f"heatmap_{output_prefix}_{hp}.csv"
        pivot.to_csv(fname)
        print(f"  {hp}:\n{pivot.to_string()}\n  → Saved to {fname}\n")

    # ── Step 6: Red flag summary ───────────────────────────────────────────────
    print(f"── Red flag summary ──")
    red_flag_records = []

    for hp in HP_COLS:
        summ = results[hp]
        pw   = pairwise_flags.get(hp, pd.DataFrame())

        sig_worse_vals = set()
        if not pw.empty:
            sig_worse_vals = set(pw.loc[pw["significantly_worse"], hp].tolist())

        for _, row in summ.iterrows():
            val         = row[hp]
            frac_bad    = row["frac_tasks_bottom25"]
            sig_worse   = val in sig_worse_vals
            is_red_flag = (frac_bad >= RED_FLAG_TASK_FRACTION) or sig_worse

            if not is_red_flag:
                continue

            bad_task_names = [
                TASK_LABELS.get(t, t) for t in ft
                if df.loc[(df[TASK_COL] == t) & (df[hp] == val),
                           "rank_within_task"].mean() < 0.25
            ]
            red_flag_records.append({
                "hyperparameter":      hp,
                "value":               val,
                "mean_rank_all":       round(row["mean_rank"], 3),
                "mean_rank_intrinsic": (round(row["mean_rank_intrinsic"], 3)
                                        if not np.isnan(row["mean_rank_intrinsic"]) else "–"),
                "mean_rank_extrinsic": (round(row["mean_rank_extrinsic"], 3)
                                        if not np.isnan(row["mean_rank_extrinsic"]) else "–"),
                "frac_tasks_bottom25": round(frac_bad, 2),
                "n_tasks_bottom25":    int(row["n_tasks_bottom25"]),
                "stat_sig_worse":      sig_worse,
                "bottom25_in_tasks":   "; ".join(bad_task_names) if bad_task_names else "–",
            })

    if red_flag_records:
        rf_df = (
            pd.DataFrame(red_flag_records)
              .sort_values(["frac_tasks_bottom25", "mean_rank_all"],
                           ascending=[False, True])
        )
        print(rf_df.to_string(index=False))
        fname = f"red_flags_{output_prefix}.csv"
        rf_df.to_csv(fname, index=False)
        print(f"\n→ Saved to {fname}")
    else:
        print("  No red-flag settings identified at the current thresholds.")

    # ── Step 7: Consistently bad in BOTH task types ───────────────────────────
    print(f"\n── Consistently bad in BOTH intrinsic AND extrinsic ──")
    for hp in HP_COLS:
        summ     = results[hp]
        both_bad = summ[
            (summ["n_intrinsic_bottom25"] >=
             len(INTRINSIC_TASKS) * RED_FLAG_TASK_FRACTION) &
            (summ["n_extrinsic_bottom25"] >=
             len(EXTRINSIC_TASKS) * RED_FLAG_TASK_FRACTION)
        ]
        if not both_bad.empty:
            print(f"  {hp}:")
            print(both_bad[[hp, "mean_rank", "mean_rank_intrinsic",
                             "mean_rank_extrinsic",
                             "frac_tasks_bottom25"]].to_string(index=False))
        else:
            print(f"  {hp}: no value consistently bottom-25 % in both task types.")

    # ── Step 8: Pairwise HP interaction red flags ──────────────────────────────
    print(f"\n── Interaction red flags (HP pairs) ──")
    for hp_a, hp_b in itertools.combinations(HP_COLS, 2):
        combo_col = f"{hp_a}__{hp_b}"
        df_tmp    = df.copy()
        df_tmp[combo_col] = (df_tmp[hp_a].astype(str)
                             + " | " + df_tmp[hp_b].astype(str))

        combo_summary = (
            df_tmp.groupby([combo_col, TASK_COL])["rank_within_task"]
              .mean().reset_index()
              .groupby(combo_col)
              .agg(
                  mean_rank        = ("rank_within_task", "mean"),
                  n_tasks          = (TASK_COL,           "nunique"),
                  n_tasks_bottom25 = ("rank_within_task",
                                      lambda x: (x < 0.25).sum()),
              )
              .assign(frac_tasks_bottom25=lambda d: d["n_tasks_bottom25"] / d["n_tasks"])
              .reset_index()
        )
        bad = combo_summary[
            combo_summary["frac_tasks_bottom25"] >= RED_FLAG_TASK_FRACTION
        ].sort_values("frac_tasks_bottom25", ascending=False)

        if not bad.empty:
            print(f"  Pair ({hp_a}, {hp_b}):")
            print(bad[[combo_col, "mean_rank",
                        "frac_tasks_bottom25", "n_tasks"]].to_string(index=False))
        else:
            print(f"  Pair ({hp_a}, {hp_b}): no red-flag combinations found.")


# ══════════════════════════════════════════════════════════════════════════════
#  RUN BOTH ANALYSES
# ══════════════════════════════════════════════════════════════════════════════

run_analysis(
    df            = df_grouped,
    label         = "Grouped (one data point per HP configuration per task)",
    output_prefix = "grouped",
)

run_analysis(
    df            = df_ungrouped,
    label         = "Ungrouped (individual model rows, within-group variance retained)",
    output_prefix = "ungrouped",
)

print("\n✓ Both analyses complete.")


Rows after all model exclusions: 320  (dropped 2)
HP configurations (groups) : 32
Individual model rows       : 1920
Tasks                       : 6


══════════════════════════════════════════════════════════════════════
  ANALYSIS: GROUPED (ONE DATA POINT PER HP CONFIGURATION PER TASK)
══════════════════════════════════════════════════════════════════════

── Statistical tests ──
  lower:
    [lower | all tasks] KW p=0.191 → no significant difference.
    [lower | intrinsic] KW p=0.786 → no significant difference.
    [lower | extrinsic] KW p=0.118 → no significant difference.
  mincount:
    [mincount | all tasks] KW p=0.0063 ✓ → pairwise tests …
    [mincount | intrinsic] KW p=0.0000 ✓ → pairwise tests …
    [mincount | extrinsic] KW p=0.0035 ✓ → pairwise tests …
  windows:
    [windows | all tasks] KW p=0.0094 ✓ → pairwise tests …
    [windows | intrinsic] KW p=0.162 → no significant difference.
    [windows | extrinsic] KW p=0.076 → no significant difference.

── Per-task rank he